# ValidationDataSet

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/ValidationDataSet.md`


Notebook source link: [ValidationDataSet.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/ValidationDataSet.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ValidationDataSet"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"ValidationDataSet: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"ValidationDataSet: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"ValidationDataSet: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"ValidationDataSet: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "p=0.01;         % bernoilli probability",
    "N=100001;       % Number of coin flips",
    "delta = 0.001;  % binsize",
    "T=N*delta;      % total time window",
    "lambda=N*p/T    % lambda*T = N*p",
    "mu = log(lambda*delta/(1-lambda*delta))",
    "for i=1:2",
    "t=linspace(0,T,N);",
    "ind=rand(1,N)<p;     %generate the coin-flip indices for heads or 1's",
    "spikeTimes = t(ind); %get time spikes based on indices",
    "nst{i}=nspikeTrain(spikeTimes,'',delta); % create neuron spike train",
    "nst{i}.setMinTime(0);",
    "nst{i}.setMaxTime(T);",
    "end",
    "nst{1}.plotISIHistogram;",
    "spikeColl=nstColl(nst); %create a nstColl - a collection of spikeTrains",
    "cov=Covariate(t,ones(length(t),1),'Baseline','s','','',{'mu'});",
    "cc=CovColl({cov}); % Gather all the covariates",
    "trial=Trial(spikeColl, cc); %Create the trial",
    "clear c;",
    "sampleRate=1000;",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "cfgColl= ConfigColl(c); %place desired configurations in a ConfigColl structure",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results{1}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);",
    "results{2}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);",
    "figure;",
    "subplot(1,2,1);results{1}.lambda.plot; hold on; plot(results{1}.lambda.time,lambda*ones(length(results{1}.lambda.time),1),'r-.','LineWidth',3);",
    "subplot(1,2,2);results{2}.lambda.plot; hold on; plot(results{2}.lambda.time,lambda*ones(length(results{2}.lambda.time),1),'r-.','LineWidth',3);",
    "p1=0.001; % bernoilli probability of process 1",
    "N1=100000; %",
    "delta = 0.001;",
    "T1=N1*delta;",
    "lambda1=N1*p1/T1    % lambda*T = N*p",
    "mu1 = log(lambda1*delta/(1-lambda1*delta))",
    "p2=0.01;  % bernoilli probability of process 1",
    "N2=100000;",
    "T2=N2*delta;",
    "lambda2=N2*p2/T2    % lambda*T = N*p",
    "mu2 = log(lambda2*delta/(1-lambda2*delta))",
    "lambdaConst = (N1*p1 + N2*p2)/(T1+T2)",
    "muConst = log(lambdaConst*delta/(1-lambdaConst*delta))",
    "for i=1:2",
    "tTot = linspace(0,(T1+T2),(N1+N2+1));",
    "t1=tTot(tTot<=T1);",
    "ind1=rand(1,N1)<p1;",
    "spikeTimes1 = t1(ind1);",
    "t2=tTot(tTot>T1);",
    "ind2=rand(1,N2)<p2;",
    "spikeTimes2 = t2(ind2);",
    "tTot = [t1'; t2'];",
    "nst{i}=nspikeTrain([spikeTimes1 spikeTimes2],'',delta);",
    "nst{i}.setMinTime(0);",
    "nst{i}.setMaxTime(max(t2));",
    "end",
    "spikeColl=nstColl(nst); %create a nstColl",
    "cov=Covariate(tTot,[ones(length(tTot),1), tTot<=max(t1), tTot>max(t1)],'Baseline','s','','',{'muConst','mu1','mu2'});",
    "cc=CovColl({cov});",
    "sampleRate=1000;",
    "trial=Trial(spikeColl, cc);",
    "clear c;",
    "c{1} = TrialConfig({{'Baseline','muConst'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu1','mu2'}},sampleRate,[],[]);",
    "c{2}.setName('Variable');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results{1}.plotResults;",
    "results{2}.plotResults;",
    "figure;",
    "subplot(1,2,1); results{1}.lambda.plot;",
    "subplot(1,2,2); results{2}.lambda.plot;",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for ValidationDataSet.")


In [ ]:
# ValidationDataSet: load MATLAB-gold trial matrix and reproduce raster/PSTH/significance summaries.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/ValidationDataSet_gold.mat"
m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
dt = float(np.asarray(m["dt_val"], dtype=float).reshape(-1)[0]); time = np.asarray(m["time_val"], dtype=float).reshape(-1)
trial_matrix = np.asarray(m["trial_matrix_val"], dtype=float); psth = np.asarray(m["psth_val"], dtype=float).reshape(-1); sem = np.asarray(m["sem_val"], dtype=float).reshape(-1)
rates, prob_mat, sig_mat = DecodingAlgorithms.compute_spike_rate_cis(spike_matrix=trial_matrix, alpha=0.05)
exp_rates = np.asarray(m["expected_rate_val"], dtype=float).reshape(-1); exp_prob = np.asarray(m["expected_prob_val"], dtype=float); exp_sig = np.asarray(m["expected_sig_val"], dtype=int)
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
for k in range(min(18, trial_matrix.shape[0])): axes[0].vlines(time[trial_matrix[k] > 0], k + 0.6, k + 1.4, linewidth=0.5)
axes[0].set_title(f"{TOPIC}: trial raster"); axes[0].set_ylabel("trial")
axes[1].plot(time, psth, color="tab:blue", linewidth=1.2); axes[1].fill_between(time, psth - sem, psth + sem, color="tab:blue", alpha=0.2); axes[1].set_ylabel("Hz"); axes[1].set_title("PSTH mean +/- SEM")
im = axes[2].imshow(prob_mat, aspect="auto", origin="lower", cmap="viridis"); axes[2].set_title("Trial-by-trial spike-rate p-values"); axes[2].set_xlabel("trial"); axes[2].set_ylabel("trial"); fig.colorbar(im, ax=axes[2], fraction=0.03, pad=0.02)
plt.tight_layout(); plt.show()
rate_err = float(np.max(np.abs(rates - exp_rates))); prob_err = float(np.max(np.abs(prob_mat - exp_prob))); sig_mismatch = float(np.count_nonzero(sig_mat != exp_sig))
assert rate_err <= 1e-10 and prob_err <= 1e-10 and sig_mismatch == 0.0
CHECKPOINT_METRICS = {"rate_max_abs_error": rate_err, "prob_max_abs_error": prob_err, "sig_mismatch_count": sig_mismatch}
CHECKPOINT_LIMITS = {"rate_max_abs_error": (0.0, 1e-10), "prob_max_abs_error": (0.0, 1e-10), "sig_mismatch_count": (0.0, 0.0)}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
